# Part A: Exploratory Data Analysis (EDA)
Reads the **clean** tables from the bucket (`clean/latest/`) and explores them: distributions, growth over time, the EV-to-charger gap by state, relationships, and anomalies.

Run the cells from top to bottom. Only **Cell 1** needs editing. At the end, the notebook saves `eda_findings.md` and the charts (`eda_charts/`) for the repo and the slides.

## Cell 1: Settings

In [ ]:
PROJECT_ID = "depa75131-514621"
BUCKET = "cnde-ev-raw-group3"
CLEAN_PATH = f"gs://{BUCKET}/clean/latest"  # or a dated folder, e.g. clean/2026-11-25
YEAR = 2025  # latest year with registrations

## Cell 2: Sign in and download the clean tables

In [ ]:
import os

IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import auth

    auth.authenticate_user()
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    !mkdir -p clean
    !gcloud storage cp {CLEAN_PATH}/*.csv clean/
DATA = "clean"
print(sorted(os.listdir(DATA)))

## Cell 3: Load the tables and set the chart style

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd


def load(name, **kw):
    return pd.read_csv(f"{DATA}/{name}.csv", low_memory=False, **kw)


station = load("station", dtype={"zip": str}, parse_dates=["open_date"])
port = load("station_port")
network = load("ev_network")
reg = load("vehicle_registration")
fuel = load("vehicle_fuel_type")
price = load("electricity_price", parse_dates=["period_month"])
state = load("state")

# Chart style: one blue for the focus, gray for context, recessive grid, no top/right spines.
BLUE, ORANGE, GRAY, INK, MUTED = "#257de0", "#f47230", "#b8b8b8", "#191919", "#666666"
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#cccccc",
        "axes.grid": True,
        "grid.color": "#eeeeee",
        "axes.axisbelow": True,
        "text.color": INK,
        "axes.labelcolor": MUTED,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
    }
)
os.makedirs("eda_charts", exist_ok=True)


def save(fig, name):
    fig.tight_layout()
    fig.savefig(f"eda_charts/{name}.png", dpi=160, bbox_inches="tight")
    plt.show()


# The supply we count: open (E), public stations, Level 2 + DC fast ports (Level 1 is too slow).
public = station[(station.status_code == "E") & (station.access_code == "public")]
pub_ports = port[port.station_id.isin(public.station_id) & port.level_code.isin(["L2", "DCFC"])]
ports_by_state = (
    pub_ports.merge(public[["station_id", "state_code"]]).groupby("state_code").port_count.sum()
)
# For EV comparisons, count only ports open by the end of YEAR (registrations are year-end counts).
public_asof = public[public.open_date <= f"{YEAR}-12-31"]
ports_asof = (
    pub_ports[pub_ports.station_id.isin(public_asof.station_id)]
    .merge(public_asof[["station_id", "state_code"]])
    .groupby("state_code")
    .port_count.sum()
)
ev = reg[(reg.fuel_type_id == 1)]  # 1 = Electric (EV)
F = {}  # findings collected for eda_findings.md
n_ports = int(pub_ports.port_count.sum())
print(f"{len(station):,} stations, {len(public):,} open public, {n_ports:,} public ports")

## 1. Where are the public chargers?
Public Level 2 and DC fast ports by state (top 15).

In [ ]:
top = ports_by_state.sort_values(ascending=False).head(15)[::-1]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top.index, top.values, color=BLUE, height=0.7)
ax.set_title("Public charging ports by state (top 15)")
ax.set_xlabel("Public Level 2 + DC fast ports")
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.grid(axis="y", visible=False)
for y, v in enumerate(top.values):
    ax.text(v, y, f" {v:,.0f}", va="center", fontsize=8, color=MUTED)
save(fig, "1_ports_by_state")
share_top = ports_by_state.sort_values(ascending=False)
F["ports_total"] = int(ports_by_state.sum())
top_share = share_top.iloc[0] / share_top.sum()
F["ports_top_state"] = (share_top.index[0], int(share_top.iloc[0]), top_share)
F["ports_top5_share"] = share_top.head(5).sum() / share_top.sum()
print(f"{share_top.index[0]} has {top_share:.0%} of public ports")
print(f"Top 5 states have {F['ports_top5_share']:.0%}")

## 2. Charger types and station size
Most stations are small. DC fast ports are a minority of all public ports.

In [ ]:
per_station = pub_ports.groupby("station_id").port_count.sum()
mix = pub_ports.groupby("level_code").port_count.sum().reindex(["L2", "DCFC"])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [3, 2]})
counts = per_station.clip(upper=20).value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color=BLUE, width=0.8)
axes[0].set_title("Ports per public station")
axes[0].set_xlabel("Ports at the station (20 = 20 or more)")
axes[0].set_ylabel("Stations")
axes[0].set_xticks([1, 2, 4, 6, 8, 10, 12, 15, 20])
axes[0].yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
axes[1].barh(["Level 2", "DC fast"], mix.values, color=[GRAY, BLUE], height=0.6)
axes[1].set_title("Public ports by charger level")
axes[1].xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
axes[1].grid(axis="y", visible=False)
for y, v in enumerate(mix.values):
    axes[1].text(v, y, f" {v:,.0f} ({v / mix.sum():.0%})", va="center", fontsize=8, color=MUTED)
save(fig, "2_station_size_and_levels")
F["median_ports"] = per_station.median()
F["share_le2"] = (per_station <= 2).mean()
F["dcfc_share"] = mix["DCFC"] / mix.sum()
print(f"Median {F['median_ports']:.0f} ports per station")
print(f"{F['share_le2']:.0%} of stations have 1-2 ports; DC fast = {F['dcfc_share']:.0%}")

## 3. Who runs the chargers?
Share of public ports by charging network (top 10).

In [ ]:
by_net = (
    pub_ports.merge(public[["station_id", "network_id"]])
    .merge(network[["network_id", "network_code"]])
    .groupby("network_code")
    .port_count.sum()
    .sort_values(ascending=False)
)
top = (by_net.head(10) / by_net.sum())[::-1]
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.barh(top.index, top.values, color=BLUE, height=0.7)
ax.set_title("Share of public ports by network (top 10)")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.grid(axis="y", visible=False)
for y, v in enumerate(top.values):
    ax.text(v, y, f" {v:.0%}", va="center", fontsize=8, color=MUTED)
save(fig, "3_networks")
F["top_network"] = (by_net.index[0], by_net.iloc[0] / by_net.sum())
F["top3_network_share"] = by_net.head(3).sum() / by_net.sum()
name, share = F["top_network"]
print(f"Largest network: {name} ({share:.0%}); top 3 = {F['top3_network_share']:.0%}")

## 4. Are chargers keeping up with EVs?
Both series are indexed to 2016 = 100 so they share one axis. Ports are counted by the year they opened (cumulative); stations that have closed are not in the API, so early years are slightly undercounted.

In [ ]:
years = range(2016, YEAR + 1)
ev_us = ev.groupby("year").vehicle_count.sum().reindex(years)
opened = pub_ports.merge(public[["station_id", "open_date"]])
opened["year"] = opened.open_date.dt.year
ports_cum = opened.groupby("year").port_count.sum().cumsum().reindex(years, method="ffill")
idx_ev, idx_ports = ev_us / ev_us.iloc[0] * 100, ports_cum / ports_cum.iloc[0] * 100
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(years), idx_ev, color=BLUE, lw=2, marker="o", ms=4)
ax.plot(list(years), idx_ports, color=ORANGE, lw=2, marker="o", ms=4)
for series, label in [(idx_ev, "EVs registered"), (idx_ports, "Public ports")]:
    end = series.iloc[-1]
    ax.text(YEAR + 0.15, end, f"{label} ({end / 100:.0f}x)", va="center", fontsize=9)
ax.set_xlim(2015.5, YEAR + 2.8)
ax.set_xticks(list(years)[::2] if len(years) % 2 else list(years)[1::2])
ax.set_title("Growth since 2016 (index, 2016 = 100)")
ax.set_ylabel("Index (2016 = 100)")
save(fig, "4_growth_index")
F["ev_growth"] = ev_us.iloc[-1] / ev_us.iloc[0]
F["port_growth"] = ports_cum.iloc[-1] / ports_cum.iloc[0]
F["ev_per_port_2016"] = ev_us.iloc[0] / ports_cum.iloc[0]
F["ev_per_port_now"] = ev_us.iloc[-1] / ports_cum.iloc[-1]
print(
    f"EVs grew {F['ev_growth']:.1f}x and public ports {F['port_growth']:.1f}x since 2016; "
    f"EVs per port went from {F['ev_per_port_2016']:.1f} to {F['ev_per_port_now']:.1f}"
)

## 5. The gap: EVs per public port by state
Higher = more EVs competing for each public charger. EVs and ports are both counted at the end of the year. Blue = the 5 largest gaps; the dashed line is the national level.

In [ ]:
g = (
    ev[ev.year == YEAR]
    .set_index("state_code")
    .vehicle_count.to_frame("evs")
    .join(ports_asof.rename("ports"))
)
g["ev_per_port"] = g.evs / g.ports
g = g.sort_values("ev_per_port")
national = g.evs.sum() / g.ports.sum()
top5 = g.tail(5).index
fig, ax = plt.subplots(figsize=(7, 9))
ax.barh(g.index, g.ev_per_port, color=[BLUE if s in top5 else GRAY for s in g.index], height=0.75)
ax.axvline(national, color=INK, lw=1, ls="--")
ax.text(national, len(g) - 0.3, f" US {national:.1f}", fontsize=8, color=INK, va="bottom")
ax.set_title(f"EVs per public charging port, {YEAR}")
ax.set_xlabel("Registered EVs per public Level 2 + DC fast port")
ax.tick_params(axis="y", labelsize=7)
ax.grid(axis="y", visible=False)
save(fig, "5_gap_by_state")
F["national_gap"] = national
F["gap_top5"] = [(s, g.loc[s, "ev_per_port"]) for s in top5[::-1]]
F["gap_bottom3"] = [(s, g.loc[s, "ev_per_port"]) for s in g.head(3).index]
display(g.sort_values("ev_per_port", ascending=False).head(10).round(1))

## 6. Is electricity price related to EV adoption?
Each dot is a state: average residential price in the year vs EVs per 1,000 registered vehicles.

In [ ]:
p = (
    price[(price.period_month.dt.year == YEAR) & (price.sector_id == "RES")]
    .groupby("state_code")
    .price_cents_kwh.mean()
)
allveh = reg[reg.year == YEAR].groupby("state_code").vehicle_count.sum()
ev_now = ev[ev.year == YEAR].set_index("state_code").vehicle_count
d = pd.DataFrame({"price": p, "ev_per_1000": ev_now / allveh * 1000}).dropna()
pearson = d.price.corr(d.ev_per_1000)
spearman = d.price.corr(d.ev_per_1000, method="spearman")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(d.price, d.ev_per_1000, s=40, color=BLUE, edgecolor="white", linewidth=1)
for s in d.ev_per_1000.nlargest(4).index.union(d.price.nlargest(2).index):
    ax.annotate(
        s,
        (d.price[s], d.ev_per_1000[s]),
        xytext=(4, 3),
        textcoords="offset points",
        fontsize=8,
        color=MUTED,
    )
ax.set_title(f"Electricity price vs EV adoption, {YEAR} (Spearman r = {spearman:.2f})")
ax.set_xlabel("Average residential price (cents per kWh)")
ax.set_ylabel("EVs per 1,000 vehicles")
save(fig, "6_price_vs_adoption")
F["corr"] = (pearson, spearman)
no_hi = d.drop(index=[s for s in ["HI"] if s in d.index])
F["corr_wo_hi"] = no_hi.corr(method="spearman").iloc[0, 1]
print(f"Pearson r = {pearson:.2f}, Spearman r = {spearman:.2f}")
print(f"Spearman r without Hawaii = {F['corr_wo_hi']:.2f}")

## 7. Anomalies
**a. California** is far ahead of every other state. **b. A counting change in 2021**: biodiesel and CNG counts jump and drop in 2021 because the source changed its method, while EV counts stay smooth. Small multiples, each with its own scale.

In [ ]:
ca_share_ev = ev[(ev.year == YEAR)].set_index("state_code").vehicle_count
F["ca_ev_share"] = ca_share_ev["CA"] / ca_share_ev.sum()
F["ca_port_share"] = ports_asof["CA"] / ports_asof.sum()
names = fuel.set_index("fuel_type_id").fuel_name
nat = reg.groupby(["year", "fuel_type_id"]).vehicle_count.sum().unstack()
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, fid in zip(axes, [1, 4, 6], strict=True):
    ax.plot(nat.index, nat[fid] / 1e6, color=BLUE if fid == 1 else ORANGE, lw=2, marker="o", ms=3)
    ax.axvline(2021, color=GRAY, lw=1, ls="--")
    ax.set_title(names[fid], fontsize=10)
    ax.set_ylabel("Millions")
fig.suptitle("Registrations by fuel: the 2021 method change (dashed)", fontweight="bold")
save(fig, "7_series_break_2021")
F["bio_2020_2021"] = (nat.loc[2020, 4], nat.loc[2021, 4])
F["cng_2020_2021"] = (nat.loc[2020, 6], nat.loc[2021, 6])
print(f"California: {F['ca_ev_share']:.0%} of US EVs, {F['ca_port_share']:.0%} of ports")

## Cell 4: Save the findings
Writes `eda_findings.md` (the numbers come from this run) and zips the charts. Both download to your computer.

In [ ]:
def pct(x):
    return f"{x:.0%}"


top5_txt = ", ".join(f"{s} ({v:.1f})" for s, v in F["gap_top5"])
low3_txt = ", ".join(f"{s} ({v:.1f})" for s, v in F["gap_bottom3"])
r = F["corr"][1]
strength = "Weak" if abs(r) < 0.3 else ("Moderate" if abs(r) < 0.6 else "Strong")
direction = "positive" if r > 0 else "negative"
if r > 0:
    corr_meaning = (
        "States with higher prices tend to have MORE EVs, likely because coastal, "
        "higher-income states have both. Price does not hold back adoption, so use it "
        "only as a cost factor in the gap score, not as a demand signal"
    )
else:
    corr_meaning = "Higher prices go with fewer EVs; use price as a cost factor in the gap score"
name, share = F["top_network"]
bio, cng = F["bio_2020_2021"], F["cng_2020_2021"]
rows = [
    (
        "Chargers are concentrated in a few states",
        f"{F['ports_top_state'][0]} has {pct(F['ports_top_state'][2])} of public ports; "
        f"the top 5 states have {pct(F['ports_top5_share'])}",
        "Compare states with a ratio (EVs per port), not raw counts",
    ),
    (
        "Most stations are small, and fast charging is a minority",
        f"Median {F['median_ports']:.0f} ports per station; {pct(F['share_le2'])} have "
        f"1-2 ports; DC fast = {pct(F['dcfc_share'])} of public ports",
        "The gap score should weigh DC fast ports separately",
    ),
    (
        "A few networks run most ports",
        f"{name} has {pct(share)}; the top 3 have {pct(F['top3_network_share'])}",
        "Network is a useful dashboard filter",
    ),
    (
        "EVs are growing faster than public chargers",
        f"From 2016 to {YEAR}, EVs grew {F['ev_growth']:.1f}x and public ports "
        f"{F['port_growth']:.1f}x; EVs per port went from {F['ev_per_port_2016']:.1f} "
        f"to {F['ev_per_port_now']:.1f}",
        "The gap is widening: this supports the business problem",
    ),
    (
        "The gap differs a lot by state",
        f"US average {F['national_gap']:.1f} EVs per port. Largest: {top5_txt}. "
        f"Smallest: {low3_txt}",
        "These are the first candidates for the top-5 recommendation",
    ),
    (
        f"{strength} {direction} relationship between electricity price and EV adoption",
        f"Pearson r = {F['corr'][0]:.2f}, Spearman r = {r:.2f} "
        f"(without Hawaii: {F['corr_wo_hi']:.2f})",
        corr_meaning,
    ),
    (
        "California is an outlier",
        f"{pct(F['ca_ev_share'])} of US EVs and {pct(F['ca_port_share'])} of public ports",
        "Show it, but use ratios so it does not dominate; consider a view without CA",
    ),
    (
        "Registration method changed in 2021",
        f"Biodiesel {bio[0]:,.0f} to {bio[1]:,.0f}; CNG {cng[0]:,.0f} to {cng[1]:,.0f} "
        "(2020 to 2021)",
        "Use only EV and PHEV counts across years",
    ),
]
lines = [
    "# EDA Findings (Part A)",
    "",
    f"Generated by `Scripts/profiling/PartA_EDA.ipynb` from `{CLEAN_PATH}`.",
    "Supply = open, public Level 2 + DC fast ports (findings 1-3: today; "
    f"findings 4-7: open by the end of {YEAR}). Demand = registered EVs at the end of {YEAR}.",
    "",
    "| # | Finding | Evidence | What it means for the project |",
    "|---|---|---|---|",
]
lines += [f"| {i} | {a} | {b} | {c} |" for i, (a, b, c) in enumerate(rows, start=1)]
lines += [
    "",
    "Limitations: closed stations are not in the API, so ports by opening year "
    "undercount early years; registration counts are rounded to the nearest 100.",
]
md = "\n".join(lines) + "\n"
with open("eda_findings.md", "w") as f:
    f.write(md)
print(md)
if IN_COLAB:
    !zip -qr eda_charts.zip eda_charts
    from google.colab import files

    files.download("eda_findings.md")
    files.download("eda_charts.zip")